# RAG PDF Research Corpus System

**Version:** 1.0  
**Date:** November 21, 2025  
**Based on:** RAG_PDF_System_Spec_v2.1  
**Platform:** Google Colab

## Description

This notebook implements a comprehensive RAG (Retrieval-Augmented Generation) system for processing and organizing academic PDFs from Google Drive. The system:

- Extracts metadata from academic PDFs (arXiv papers, journal articles)
- Generates AI-powered summaries using GPT-5.1 with reasoning
- Builds a 3-tier hierarchical topic taxonomy through clustering
- Creates vector embeddings and FAISS index for efficient retrieval
- Enables natural language querying with RAG-based responses
- Uses LangGraph for workflow orchestration

## Attribution

Developed for organizing research corpus from arXiv.org and other academic sources.  
Repository: https://github.com/dhar174/research_corpus_organizer

---
# Section 1: Environment Setup and Configuration

## 1.1 Environment Verification

In [ ]:
import sys
import platform

print("=" * 50)
print("Python Environment Check")
print("=" * 50)

# Check Python version
python_version = sys.version_info
print(f"Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")

if python_version >= (3, 10):
    print("✓ Python version is compatible (3.10+)")
else:
    print("✗ WARNING: Python 3.10+ is required for optimal compatibility")
    print(f"  Current version: {python_version.major}.{python_version.minor}")

print(f"\nPlatform: {platform.platform()}")
print(f"Architecture: {platform.machine()}")

In [ ]:
import subprocess

print("=" * 50)
print("GPU/CPU Availability Check")
print("=" * 50)

# Check for NVIDIA GPU
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True, timeout=5)
    if result.returncode == 0 and result.stdout.strip():
        print("✓ GPU Available:")
        for line in result.stdout.strip().split('\n'):
            print(f"  {line}")
    else:
        print("○ No NVIDIA GPU detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("○ No NVIDIA GPU detected (nvidia-smi not available)")

# Check CPU info
try:
    import multiprocessing
    cpu_count = multiprocessing.cpu_count()
    print(f"\n✓ CPU Cores Available: {cpu_count}")
except:
    print("\n○ Unable to determine CPU count")

print("\nNote: This notebook uses FAISS-CPU for vector indexing.")
print("GPU is not required but may speed up some operations.")

In [ ]:
import os
from datetime import datetime

print("=" * 50)
print("Runtime Information")
print("=" * 50)

print(f"Current Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Working Directory: {os.getcwd()}")

# Check if running in Colab
try:
    import google.colab
    print("\n✓ Running in Google Colab")
    IN_COLAB = True
except ImportError:
    print("\n○ Not running in Google Colab")
    IN_COLAB = False

# Display memory info
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"\nMemory Available: {mem.available / (1024**3):.2f} GB")
    print(f"Memory Total: {mem.total / (1024**3):.2f} GB")
except ImportError:
    print("\n○ Install psutil to view memory information")

## 1.2 Install Dependencies

Installing all required packages for the RAG PDF Research Corpus System.  
This may take several minutes on first run.

In [ ]:
%%capture --no-display
# Suppress installation output; remove %%capture to see detailed progress

import sys

print("Installing dependencies...")
print("This may take 3-5 minutes on first run.")
print("-" * 50)

# Core dependencies for RAG and LangGraph
!{sys.executable} -m pip install --upgrade pip -q
!{sys.executable} -m pip install openai>=1.3.0 -q
!{sys.executable} -m pip install langgraph>=0.0.30 -q
!{sys.executable} -m pip install langchain>=0.1.0 -q

# PDF processing
!{sys.executable} -m pip install pymupdf>=1.23.0 -q

# Vector indexing and clustering
!{sys.executable} -m pip install faiss-cpu>=1.7.4 -q
!{sys.executable} -m pip install scikit-learn>=1.3.0 -q
!{sys.executable} -m pip install hdbscan>=0.8.33 -q

# Data handling and utilities
!{sys.executable} -m pip install pandas>=2.0.0 -q
!{sys.executable} -m pip install numpy>=1.24.0 -q
!{sys.executable} -m pip install tqdm>=4.65.0 -q

# Visualization
!{sys.executable} -m pip install matplotlib>=3.7.0 -q
!{sys.executable} -m pip install seaborn>=0.12.0 -q

# API access and date parsing
!{sys.executable} -m pip install python-dateutil>=2.8.2 -q
!{sys.executable} -m pip install requests>=2.31.0 -q

# Optional: OCR fallback for scanned PDFs
!{sys.executable} -m pip install pytesseract>=0.3.10 -q
!{sys.executable} -m pip install Pillow>=10.0.0 -q

# Data validation
!{sys.executable} -m pip install pydantic>=2.0.0 -q

print("✓ All dependencies installed successfully!")
print("\nNote: If you encounter import errors, restart the runtime:")
print("  Runtime > Restart runtime")

## 1.3 Import Statements

Importing all required libraries and verifying successful installation.

In [ ]:
# Standard library imports
import os
import sys
import json
import hashlib
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, TypedDict, Any
from collections import defaultdict

# Third-party imports - Core AI/ML
try:
    import openai
    from openai import OpenAI
    print("✓ OpenAI SDK imported successfully")
except ImportError as e:
    print(f"✗ Failed to import OpenAI: {e}")
    raise

try:
    import langgraph
    from langgraph.graph import StateGraph
    print("✓ LangGraph imported successfully")
except ImportError as e:
    print(f"✗ Failed to import LangGraph: {e}")
    raise

# Third-party imports - PDF processing
try:
    import fitz  # PyMuPDF
    print("✓ PyMuPDF (fitz) imported successfully")
except ImportError as e:
    print(f"✗ Failed to import PyMuPDF: {e}")
    raise

# Third-party imports - Vector indexing and ML
try:
    import faiss
    print("✓ FAISS imported successfully")
except ImportError as e:
    print(f"✗ Failed to import FAISS: {e}")
    raise

try:
    import numpy as np
    from sklearn.cluster import AgglomerativeClustering, KMeans
    from sklearn.metrics import silhouette_score
    print("✓ NumPy and scikit-learn imported successfully")
except ImportError as e:
    print(f"✗ Failed to import NumPy/scikit-learn: {e}")
    raise

# Optional: HDBSCAN for density-based clustering
try:
    import hdbscan
    print("✓ HDBSCAN imported successfully")
    HDBSCAN_AVAILABLE = True
except ImportError:
    print("○ HDBSCAN not available (optional)")
    HDBSCAN_AVAILABLE = False

# Third-party imports - Data handling
try:
    import pandas as pd
    from tqdm.auto import tqdm
    print("✓ Pandas and tqdm imported successfully")
except ImportError as e:
    print(f"✗ Failed to import data handling libraries: {e}")
    raise

# Third-party imports - Visualization
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    print("✓ Matplotlib and Seaborn imported successfully")
except ImportError as e:
    print(f"✗ Failed to import visualization libraries: {e}")
    raise

# Third-party imports - Utilities
try:
    from dateutil import parser as date_parser
    import requests
    print("✓ dateutil and requests imported successfully")
except ImportError as e:
    print(f"✗ Failed to import utility libraries: {e}")
    raise

# Optional: OCR dependencies
try:
    import pytesseract
    from PIL import Image
    print("✓ Tesseract OCR and Pillow imported successfully")
    OCR_AVAILABLE = True
except ImportError:
    print("○ OCR libraries not available (optional)")
    OCR_AVAILABLE = False

# Third-party imports - Data validation
try:
    from pydantic import BaseModel, Field, validator
    print("✓ Pydantic imported successfully")
except ImportError as e:
    print(f"✗ Failed to import Pydantic: {e}")
    raise

print("\n" + "=" * 50)
print("All required imports successful!")
print("=" * 50)

## 1.4 Configuration

Configure the RAG PDF Research Corpus System parameters.  
Edit the values below to customize the pipeline behavior.

In [ ]:
from typing import Optional, Literal
from pydantic import BaseModel, Field, field_validator

class RunConfig(BaseModel):
    """Configuration for the RAG PDF Research Corpus System."""
    
    # Google Drive Configuration
    drive_folder_path: str = Field(
        default="/content/drive/MyDrive/research_papers",
        description="Absolute path to the Google Drive folder containing PDFs"
    )
    
    # OpenAI API Configuration
    openai_api_key: str = Field(
        default="",
        description="OpenAI API key (will prompt for secure input if empty)"
    )
    
    # Model Configuration
    summary_model: str = Field(
        default="gpt-4-turbo",
        description="Model for generating paper summaries (e.g., gpt-4-turbo, gpt-5.1)"
    )
    
    taxonomy_model: str = Field(
        default="gpt-4-turbo",
        description="Model for generating topic labels and descriptions"
    )
    
    classification_model: str = Field(
        default="gpt-4-turbo",
        description="Model for classifying papers into taxonomy"
    )
    
    embedding_model: str = Field(
        default="text-embedding-3-small",
        description="Model for generating embeddings"
    )
    
    # Reasoning effort levels (for GPT-5.1 if available)
    summary_reasoning_effort: Literal["low", "medium", "high"] = Field(
        default="medium",
        description="Reasoning effort for summarization"
    )
    
    taxonomy_reasoning_effort: Literal["low", "medium", "high"] = Field(
        default="high",
        description="Reasoning effort for taxonomy generation"
    )
    
    classification_reasoning_effort: Literal["low", "medium", "high"] = Field(
        default="medium",
        description="Reasoning effort for classification"
    )
    
    # Chunking Configuration
    chunk_size: int = Field(
        default=1500,
        description="Target size for text chunks in characters",
        ge=500,
        le=5000
    )
    
    chunk_overlap: int = Field(
        default=200,
        description="Overlap between chunks in characters",
        ge=0,
        le=1000
    )
    
    max_chunks_per_paper: int = Field(
        default=50,
        description="Maximum number of chunks per paper (0 = unlimited)",
        ge=0
    )
    
    # Token Limits
    max_summary_tokens: int = Field(
        default=500,
        description="Maximum tokens for generated summaries",
        ge=100,
        le=4000
    )
    
    max_context_tokens: int = Field(
        default=8000,
        description="Maximum context tokens for API calls",
        ge=1000,
        le=128000
    )
    
    # Clustering Configuration
    tier1_target_k: Optional[int] = Field(
        default=None,
        description="Target number of Tier 1 topics (None = auto-determine)"
    )
    
    tier2_target_k: Optional[int] = Field(
        default=None,
        description="Target number of Tier 2 topics per Tier 1 cluster (None = auto)"
    )
    
    tier3_target_k: Optional[int] = Field(
        default=None,
        description="Target number of Tier 3 topics per Tier 2 cluster (None = auto)"
    )
    
    clustering_algorithm: Literal["kmeans", "agglomerative", "hdbscan"] = Field(
        default="agglomerative",
        description="Clustering algorithm to use"
    )
    
    # Feature Flags
    enable_ocr: bool = Field(
        default=False,
        description="Enable OCR fallback for scanned PDFs"
    )
    
    enable_deep_analysis_pass: bool = Field(
        default=False,
        description="Enable deep analysis pass (Pass 2) for detailed methodology extraction"
    )
    
    enable_metadata_extraction: bool = Field(
        default=True,
        description="Enable metadata extraction from arXiv/CrossRef APIs"
    )
    
    # Processing Configuration
    batch_size: int = Field(
        default=10,
        description="Number of papers to process in each batch",
        ge=1,
        le=100
    )
    
    max_retries: int = Field(
        default=3,
        description="Maximum number of retries for failed API calls",
        ge=0,
        le=10
    )
    
    # Output Configuration
    output_dir: str = Field(
        default="/content/drive/MyDrive/rag_output",
        description="Directory for saving outputs (CSV, FAISS index, etc.)"
    )
    
    export_format: Literal["csv", "parquet", "both"] = Field(
        default="csv",
        description="Format for exporting paper records"
    )
    
    @field_validator('chunk_overlap')
    @classmethod
    def validate_chunk_overlap(cls, v: int, info) -> int:
        """Ensure chunk overlap is less than chunk size."""
        if 'chunk_size' in info.data and v >= info.data['chunk_size']:
            raise ValueError("chunk_overlap must be less than chunk_size")
        return v
    
    @field_validator('clustering_algorithm')
    @classmethod
    def validate_clustering_algorithm(cls, v: str) -> str:
        """Validate clustering algorithm availability."""
        if v == "hdbscan" and not HDBSCAN_AVAILABLE:
            raise ValueError("HDBSCAN selected but not installed. Install with: pip install hdbscan")
        return v

print("✓ RunConfig class defined successfully")

In [ ]:
# ============================================================
# USER CONFIGURATION SECTION
# Edit the values below to customize your pipeline
# ============================================================

# Create configuration with default or custom values
config = RunConfig(
    # Google Drive folder containing your PDF files
    drive_folder_path="/content/drive/MyDrive/research_papers",
    
    # OpenAI API key (leave empty to prompt for secure input)
    openai_api_key="",
    
    # Model selections (use gpt-5.1 when available)
    summary_model="gpt-4-turbo",
    taxonomy_model="gpt-4-turbo",
    classification_model="gpt-4-turbo",
    embedding_model="text-embedding-3-small",
    
    # Reasoning effort (low/medium/high)
    summary_reasoning_effort="medium",
    taxonomy_reasoning_effort="high",
    classification_reasoning_effort="medium",
    
    # Chunking parameters
    chunk_size=1500,
    chunk_overlap=200,
    max_chunks_per_paper=50,
    
    # Clustering (None = auto-determine optimal number)
    tier1_target_k=None,
    tier2_target_k=None,
    tier3_target_k=None,
    clustering_algorithm="agglomerative",
    
    # Feature flags
    enable_ocr=False,
    enable_deep_analysis_pass=False,
    enable_metadata_extraction=True,
    
    # Output directory
    output_dir="/content/drive/MyDrive/rag_output",
    export_format="csv"
)

print("=" * 50)
print("Configuration Summary")
print("=" * 50)
print(f"Drive Folder: {config.drive_folder_path}")
print(f"Output Directory: {config.output_dir}")
print(f"Summary Model: {config.summary_model}")
print(f"Embedding Model: {config.embedding_model}")
print(f"Chunk Size: {config.chunk_size} chars")
print(f"Clustering: {config.clustering_algorithm}")
print(f"OCR Enabled: {config.enable_ocr}")
print(f"Deep Analysis: {config.enable_deep_analysis_pass}")
print("=" * 50)

# Prompt for API key if not provided
if not config.openai_api_key:
    print("\n⚠ OpenAI API key not configured.")
    print("You will be prompted to enter it securely when needed.")
else:
    print("\n✓ OpenAI API key configured")
    # Initialize OpenAI client
    client = OpenAI(api_key=config.openai_api_key)
    print("✓ OpenAI client initialized")

---
# Section 2: Data Models and Schema Definitions

*To be implemented in Phase 1*

---
# Section 3: Google Drive Integration

*To be implemented in Phase 2*

---
# Section 4: PDF Parsing and Chunking

*To be implemented in Phase 3*

---
# Section 5: Metadata Extraction

*To be implemented in Phase 4*

---
# Section 6: Embedding Generation and FAISS Index

*To be implemented in Phase 5*

---
# Section 7: Summarization (Pass 1)

*To be implemented in Phase 6*

---
# Section 8: Initial CSV Export

*To be implemented in Phase 7*

---
# Section 9: Topic Modeling and Taxonomy Construction

*To be implemented in Phase 8*

---
# Section 10: Taxonomy Review and Approval

*To be implemented in Phase 9*

---
# Section 11: Final Topic Classification (Pass 3)

*To be implemented in Phase 10*

---
# Section 12: Deep Analysis Pass (Optional - Pass 2)

*To be implemented in Phase 11*

---
# Section 13: Final CSV/Parquet Export

*To be implemented in Phase 12*

---
# Section 14: LangGraph Workflow Integration

*To be implemented in Phase 13*

---
# Section 15: Quality Control and Validation

*To be implemented in Phase 14*

---
# Section 16: RAG Query Interface

*To be implemented in Phase 15*

---
# Section 17: Utility Functions and Tools

*To be implemented in Phase 16*

---
# End of Notebook

**Next Steps:**
1. Mount Google Drive when ready to process PDFs
2. Configure your OpenAI API key
3. Proceed with Phase 1 implementation (Data Models)

For questions or issues, refer to the repository:  
https://github.com/dhar174/research_corpus_organizer